# ML Model Deployment: Concepts & Patterns

## Why Deployment Matters
A model that isn't deployed creates no value. Deployment bridges the gap between experimentation and production.

## Deployment Modes
| Mode | Latency | Throughput | Use Case |
|------|---------|------------|----------|
| **Real-time (online)** | Low (<100ms) | Moderate | Fraud detection, recommendations |
| **Batch** | High (minutes-hours) | Very high | Nightly scoring, report generation |
| **Streaming** | Medium | High | Kafka-driven pipelines |
| **Edge** | Very low | Low | Mobile apps, IoT |

## Model Serialization Formats
| Format | Framework | Pros | Cons |
|--------|-----------|------|------|
| **pickle/joblib** | scikit-learn | Simple | Python-only, security risks |
| **ONNX** | Framework-agnostic | Portable, optimized | Not all ops supported |
| **TorchScript** | PyTorch | C++ deployable, optimized | Complex API |
| **SavedModel** | TensorFlow | Full TF ecosystem | TF-only |
| **GGUF** | llama.cpp | Quantized LLMs | LLM-specific |

In [1]:
import numpy as np
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Train a pipeline
iris = load_iris()
pipeline = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
pipeline.fit(iris.data, iris.target)

# Pickle / Joblib serialization
joblib.dump(pipeline, '/tmp/model.joblib', compress=3)  # compress=3 reduces file size
print('Model saved with joblib')

# Load and predict
loaded_model = joblib.load('/tmp/model.joblib')
y_pred = loaded_model.predict(iris.data[:5])
print(f'Predictions: {y_pred}')

Model saved with joblib
Predictions: [0 0 0 0 0]


In [2]:
# ONNX Export
import torch
import torch.nn as nn
import torch.onnx

class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model = SimpleClassifier()
model.eval()

# Train briefly
X = torch.FloatTensor(iris.data)
y = torch.LongTensor(iris.target)
optimizer = torch.optim.Adam(model.parameters())
for _ in range(100):
    pred = model(X)
    loss = nn.CrossEntropyLoss()(pred, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

# Export to ONNX
dummy_input = torch.randn(1, 4)  # batch_size=1, features=4
torch.onnx.export(
    model, dummy_input, '/tmp/model.onnx',
    input_names=['features'],
    output_names=['logits'],
    dynamic_axes={'features': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=17
)
print('ONNX model exported')

/tmp/ipykernel_159913/1991344514.py:31: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0619 17:00:26.066000 159913 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model exported


In [3]:
# ONNX Runtime inference
try:
    import onnxruntime as ort
    import numpy as np

    # Load ONNX model
    ort_session = ort.InferenceSession('/tmp/model.onnx')

    # Print model info
    print('Inputs:', [inp.name for inp in ort_session.get_inputs()])
    print('Outputs:', [out.name for out in ort_session.get_outputs()])

    # Run inference
    input_data = np.float32(iris.data[:5])
    outputs = ort_session.run(None, {'features': input_data})
    logits = outputs[0]
    predictions = np.argmax(logits, axis=1)
    print(f'ONNX predictions: {predictions}')
except ImportError:
    print('Install onnxruntime: pip install onnxruntime')

Inputs: ['features']
Outputs: ['logits']
ONNX predictions: [0 0 0 0 0]


In [4]:
# TorchScript two approaches

# 1. Tracing (records operations with a sample input)
scripted_model = torch.jit.trace(model, dummy_input)
torch.jit.save(scripted_model, '/tmp/model_traced.pt')
print('Traced TorchScript model saved')

# 2. Scripting (analyzes code statically handles control flow)
# @torch.jit.script  # decorator
# def my_function(x: torch.Tensor) -> torch.Tensor:
#     if x.sum() > 0:
#         return x * 2
#     return x

# Load TorchScript model (can be used in C++ without Python)
loaded = torch.jit.load('/tmp/model_traced.pt')
with torch.no_grad():
    out = loaded(torch.FloatTensor(iris.data[:3]))
    print(f'TorchScript predictions: {out.argmax(1)}')

Traced TorchScript model saved
TorchScript predictions: tensor([0, 0, 0])


## Serving Frameworks

### TorchServe
```bash
# Package model
torch-model-archiver --model-name iris_classifier \
  --version 1.0 \
  --serialized-file model_traced.pt \
  --handler image_classifier  # or custom handler

# Start server
torchserve --start --model-store model_store --models iris=iris_classifier.mar

# Inference
curl -X POST http://localhost:8080/predictions/iris \
  -H 'Content-Type: application/json' \
  -d '[1.5, 2.5, 3.5, 4.5]'
```

### Triton Inference Server
```bash
# Model repository structure
model_repository/
  iris_onnx/
    config.pbtxt     # model configuration
    1/
      model.onnx

# config.pbtxt
name: "iris_onnx"
backend: "onnxruntime"
max_batch_size: 64
input [
  { name: "features", data_type: TYPE_FP32, dims: [4] }
]
output [
  { name: "logits", data_type: TYPE_FP32, dims: [3] }
]

# Start Triton
docker run --gpus=1 --rm -p8000:8000 -p8001:8001 \
  -v /path/to/model_repository:/models \
  nvcr.io/nvidia/tritonserver:24.01-py3 \
  tritonserver --model-repository=/models
```

## Deployment Strategies

### Blue-Green Deployment
```
Load Balancer → [Blue (v1.0) 100%]
                [Green (v1.1) 0%]

# Switch traffic:
Load Balancer → [Blue (v1.0) 0%]
                [Green (v1.1) 100%]
```

### Canary Release
```
Load Balancer → [Stable (v1.0) 95%]
                [Canary (v1.1) 5%]
# Gradually increase canary traffic if metrics are good
```

### Shadow Mode
```
All requests → [Production (v1.0)] serves response
            → [Shadow (v1.1)] predictions logged, not served
# Compare shadow vs production predictions for validation
```

## Additional Learning Resources

### Documentation
- [ONNX Docs](https://onnx.ai/)
- [ONNX Runtime](https://onnxruntime.ai/docs/)
- [TorchServe Docs](https://pytorch.org/serve/)
- [Triton Inference Server](https://developer.nvidia.com/triton-inference-server)
- [TensorFlow Serving](https://www.tensorflow.org/tfx/guide/serving)

### Books
- [Designing Machine Learning Systems Chip Huyen](https://www.oreilly.com/library/view/designing-machine-learning/9781098107958/)
- [Building Machine Learning Powered Applications Emmanuel Ameisen](https://www.oreilly.com/library/view/building-machine-learning/9781492045106/)

### Papers
- [Clipper: A Low-Latency Online Prediction Serving System](https://www.usenix.org/system/files/conference/nsdi17/nsdi17-crankshaw.pdf)